What You're Aiming For
In this checkpoint, we are going to work on the 'Electric Vehicle Data' dataset that was provided by Kaggle as part of the Electric Vehicle Price Prediction competition.

Dataset description: This dataset contains information on the Battery Electric Vehicles (BEVs) and Plug-in Hybrid Electric Vehicles (PHEVs) that are currently registered with the Washington State Department of Licensing (DOL). This dataset was introduced as part of an official invitation-based competition on Kaggle. Our SVM model should answer the question "This is my car's model & make, along with a few other parameters, what price can this vehicle be brought or sold?”

➡️ Dataset link

https://i.imgur.com/IpuCW3s.jpg

➡️Columns explanation 

 


Instructions
Import you data and perform basic data exploration phase
Display general information about the dataset
Create a pandas profiling reports to gain insights into the dataset
Handle Missing and corrupted values
Remove duplicates, if they exist
Handle outliers, if they exist
Encode categorical features
Select your target variable and the features
Split your dataset to training and test sets
Build and train an SVM model on the training set
Assess your model performance on the test set using relevant evaluation metrics
Discuss with your cohort alternative ways to improve your model performance

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('Electric_cars_dataset.csv')

# Data Cleaning

# First clean the target column - convert to numeric and handle non-numeric values
df['Expected Price ($1k)'] = pd.to_numeric(df['Expected Price ($1k)'], errors='coerce')

# Now remove rows where target is NA (including those that couldn't be converted)
df = df.dropna(subset=['Expected Price ($1k)'])

# Remove duplicates
df = df.drop_duplicates()
print("\nShape after removing duplicates and cleaning target:", df.shape)

# Handle outliers in numerical features (using IQR method for simplicity)
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numerical_cols:
    if col != 'Expected Price ($1k)':  # Don't remove outliers from target variable
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

print("\nShape after handling outliers:", df.shape)

# Feature selection - we'll select relevant features for our model
features = ['Make', 'Model', 'Model Year', 'Electric Vehicle Type', 'Electric Range', 'Base MSRP', 'Legislative District']
target = 'Expected Price ($1k)'

# Filter the dataframe
df = df[features + [target]]

# Preprocessing pipeline

# Identify categorical and numerical features
categorical_features = ['Make', 'Model', 'Electric Vehicle Type']
numerical_features = ['Model Year', 'Electric Range', 'Base MSRP', 'Legislative District']

# Create transformers
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)])

# Split data into train and test sets
X = df.drop(target, axis=1)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train SVM model
svm_model = Pipeline(steps=[('preprocessor', preprocessor),
                          ('regressor', SVR(kernel='rbf', C=1.0, epsilon=0.1))])

svm_model.fit(X_train, y_train)

# Evaluate the model
y_pred = svm_model.predict(X_test)

print("\nModel Evaluation:")
print("Mean Absolute Error (MAE):", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_pred))
print("Root Mean Squared Error (RMSE):", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R-squared (R2):", r2_score(y_test, y_pred))
